# Phase 3 — StreamFlix Business KPI Calculations

Ten leadership KPIs are calculated from the supplied StreamFlix tables using the formulas in the project brief.

## 1. Load data

In [1]:
import pandas as pd, numpy as np
from pathlib import Path

DATA_DIR = Path(".")  # change if the CSV files are stored elsewhere

subscribers = pd.read_csv(DATA_DIR/"subscribers.csv", parse_dates=["signup_date","churn_date"])
titles = pd.read_csv(DATA_DIR/"titles.csv", parse_dates=["date_added","license_expiry"])
watch_history = pd.read_csv(DATA_DIR/"watch_history.csv", parse_dates=["watch_date"])
ratings = pd.read_csv(DATA_DIR/"ratings.csv", parse_dates=["rating_date"])
reviews = pd.read_csv(DATA_DIR/"reviews.csv", parse_dates=["review_date"])
watchlist = pd.read_csv(DATA_DIR/"watchlist.csv", parse_dates=["added_date"])

tables = {
    "subscribers": subscribers,
    "titles": titles,
    "watch_history": watch_history,
    "ratings": ratings,
    "reviews": reviews,
    "watchlist": watchlist
}

## 2. KPI calculations

In [2]:
watch_hours = watch_history.watch_duration_min.sum()/60
active_subs = subscribers.is_active.sum()
total_subs = len(subscribers)
active_rate = active_subs/total_subs*100
churn_rate = (~subscribers.is_active).sum()/total_subs*100
avg_completion = watch_history.completion_pct.mean()
mrr = subscribers.loc[subscribers.is_active,'monthly_price_usd'].sum()
arpu = mrr/active_subs
avg_watch_time = watch_hours/active_subs
watchlist_conversion = watchlist.watched.mean()*100
plays = titles[['title_id','total_plays']].sort_values('total_plays',ascending=False)
top10n = int(np.ceil(len(plays)*0.10))
hit_concentration = plays.head(top10n).total_plays.sum()/plays.total_plays.sum()*100
original_ids = set(titles.loc[titles.is_original,'title_id'])
original_hours = watch_history.loc[watch_history.title_id.isin(original_ids),'watch_duration_min'].sum()/60
original_share = original_hours/watch_hours*100
kpis = pd.DataFrame({
 'KPI':['Total Watch Hours','Active Rate','Churn Rate','Avg Completion Rate','MRR','ARPU','Avg Watch Time / Active Subscriber','Watchlist Conversion','Hit Concentration','Originals Share of Hours'],
 'Value':[watch_hours,active_rate,churn_rate,avg_completion,mrr,arpu,avg_watch_time,watchlist_conversion,hit_concentration,original_share]
})
kpis

,KPI,Value
0,Total Watch Hours,3.333468e+06
1,Active Rate,7.466000e+01
2,Churn Rate,2.534000e+01
3,Avg Completion Rate,6.531071e+01
4,MRR,1.758685e+05
5,ARPU,1.570395e+01
6,Avg Watch Time / Active Subscriber,2.976576e+02
7,Watchlist Conversion,4.622769e+01
8,Hit Concentration,3.098723e+01
9,Originals Share of Hours,2.716876e+01


## 3. Content and subscriber insights

In [3]:
wh = watch_history.merge(titles[['title_id','title_name','primary_genre','country','type','language','is_original']],on='title_id',how='left')
print('Top genre by watch hours:')
display(wh.groupby('primary_genre').watch_duration_min.sum().div(60).sort_values(ascending=False).head())
print('Top countries by watch hours:')
display(wh.groupby('country').watch_duration_min.sum().div(60).sort_values(ascending=False).head(10))
print('Plan distribution:')
display(subscribers.plan_type.value_counts())

Top genre by watch hours:


primary_genre
Drama          501884.466667
Comedy         423149.053333
Action         393171.916667
Thriller       307024.663333
Documentary    291216.898333
Name: watch_duration_min, dtype: float64

Top countries by watch hours:


country
United States     822684.680000
India             464337.756667
United Kingdom    282946.390000
South Korea       223476.403333
Japan             207130.865000
Brazil            176353.601667
Mexico            157176.858333
Spain             143656.396667
France            115011.761667
Canada            107072.951667
Name: watch_duration_min, dtype: float64

Plan distribution:


plan_type
Standard          6100
Basic with Ads    4458
Premium           4442
Name: count, dtype: int64

## 4. SQL cross-check
The supplied SQL file contains 12 analytical queries covering genre, country, release year, content type, language, completion, plan engagement, device, churn, MRR/ARPU, watchlist conversion and investment efficiency. The Python KPIs above follow the project brief formulas.

## 5. Business interpretation
Use the target benchmarks in the project brief: Active Rate >70%, Churn Rate <30%, Average Completion >60%, and Watchlist Conversion >40%. The supplied dataset clears all four benchmarks.